In [54]:
import pandas as pd

In [55]:
df=pd.read_csv('/content/drive/MyDrive/complaints_extended.csv')

In [56]:
df.value_counts('sector')

,count
sector,
road,478
sanitation,438
water,401
electricity,383
parks,164
building,136


In [57]:
df.drop(['municipal_corp','date','month','year'],axis=1,inplace=True)

In [58]:
df

,text,sector,severity
0,Broken park bench at Central Park in MMC juris...,parks,0
1,Bridge approach damaged near West Zone in MMC ...,road,2
2,Water bill discrepancy for Ward 2,water,1
3,Drain blockage at Central Zone in RMC jurisdic...,water,1
4,Building material on road in WMC jurisdiction,building,1
...,...,...,...
1995,Water meter not working in VMC jurisdiction,water,0
1996,Tree needs pruning in GMC jurisdiction,parks,0
1997,Water quality issues in Industrial Area,water,2
1998,Bridge approach damaged near Ward 5 in KMC jur...,road,2


In [59]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, GRU, Dense, Dropout


In [60]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z ]", "", text)
    return text

df["text"] = df["text"].apply(clean_text)



In [61]:
sector_encoder = LabelEncoder()
df["sector_encoded"] = sector_encoder.fit_transform(df["sector"])

severity_encoder = LabelEncoder()
df["severity_encoded"] = severity_encoder.fit_transform(df["severity"])


In [62]:
MAX_WORDS = 10000
MAX_LEN = 50

tokenizer = Tokenizer(num_words=MAX_WORDS)
tokenizer.fit_on_texts(df["text"])

X = tokenizer.texts_to_sequences(df["text"])
X = pad_sequences(X, maxlen=MAX_LEN)

y_sector = df["sector_encoded"]
y_severity = df["severity_encoded"]


In [63]:
X_train, X_test, y_sec_train, y_sec_test, y_sev_train, y_sev_test = train_test_split(
    X, y_sector, y_severity, test_size=0.2, random_state=42
)


In [64]:
input_layer = Input(shape=(MAX_LEN,))

embedding = Embedding(input_dim=MAX_WORDS, output_dim=128)(input_layer)

shared_lstm = LSTM(128, return_sequences=False)(embedding)
shared_dropout = Dropout(0.3)(shared_lstm)

# Output 1: Severity
severity_output = Dense(3, activation="softmax", name="severity")(shared_dropout)

# Output 2: Sector
sector_output = Dense(len(sector_encoder.classes_), activation="softmax", name="sector")(shared_dropout)

model = Model(inputs=input_layer, outputs=[severity_output, sector_output])

model.compile(
    optimizer="adam",
    loss={
        "severity": "sparse_categorical_crossentropy",
        "sector": "sparse_categorical_crossentropy"
    },
    metrics={
        "severity": "accuracy",
        "sector": "accuracy"
    }
)

model.summary()


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 50)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, 50, 128)   │  1,280,000 │ input_layer_2[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ (None, 128)       │    131,584 │ embedding_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 128)       │          0 │ lstm_2[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ severity (Dense)    │ (None, 3)         │        387 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sector (Dense)      │ (None, 6)         │        774 │ dropout_2[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,412,745 (5.39 MB)

 Trainable params: 1,412,745 (5.39 MB)

 Non-trainable params: 0 (0.00 B)

In [65]:
history = model.fit(
    X_train,
    {"severity": y_sev_train, "sector": y_sec_train},
    validation_data=(X_test, {"severity": y_sev_test, "sector": y_sec_test}),
    epochs=10,
    batch_size=32
)


Epoch 1/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 9s 128ms/step - loss: 2.7780 - sector_accuracy: 0.2995 - sector_loss: 1.6962 - severity_accuracy: 0.4143 - severity_loss: 1.0818 - val_loss: 2.2024 - val_sector_accuracy: 0.5925 - val_sector_loss: 1.1888 - val_severity_accuracy: 0.6050 - val_severity_loss: 1.0059
Epoch 2/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 7s 133ms/step - loss: 2.2760 - sector_accuracy: 0.5972 - sector_loss: 1.2810 - severity_accuracy: 0.5448 - severity_loss: 0.9950 - val_loss: 1.5238 - val_sector_accuracy: 0.7900 - val_sector_loss: 0.7434 - val_severity_accuracy: 0.6900 - val_severity_loss: 0.7758
Epoch 3/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 12s 177ms/step - loss: 1.2188 - sector_accuracy: 0.8020 - sector_loss: 0.5598 - severity_accuracy: 0.7127 - severity_loss: 0.6591 - val_loss: 0.5752 - val_sector_accuracy: 0.8875 - val_sector_loss: 0.2592 - val_severity_accuracy: 0.8775 - val_severity_loss: 0.3133
Epoch 4/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 7s 135ms/step - loss: 0.4781 - sector_accuracy: 0.9324 

In [66]:
model.save('NLP_Severity_&_Sector.h5')ffdbd jylgkjhb

In [68]:
def predict_issue(text):
    text = clean_text(text)
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=MAX_LEN)

    sev_pred, sec_pred = model.predict(pad)

    severity = severity_encoder.inverse_transform([np.argmax(sev_pred)])
    sector = sector_encoder.inverse_transform([np.argmax(sec_pred)])

    return {
        "severity": severity[0],
        "sector": sector[0]
    }

# Test
print(predict_issue("How much i love ved sawant"))


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 223ms/step
{'severity': np.int64(0), 'sector': 'parks'}
